# KV260 小鼠全身解剖重建
默认使用 Digimouse 的 16 层解剖标注。数据及派生模型遵守 CC BY-SA 4.0。使用 Kria-PYNQ 的 Python 内核。板测单元格会加载 overlay，覆盖当前 PL。请先停止其他 PL 程序。没有连接 KV260 时只执行预览单元格。参考场景来自 CPU，实板验收记录由目标板生成。

In [ ]:
from pathlib import Path
import os, sys, subprocess
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'start.sh').is_file(), '请从工程根目录或 notebooks 目录启动'
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'software'))
print('工程:', ROOT, 'Python:', sys.executable)
subprocess.run([sys.executable, 'scripts/verify_bundle.py', '--scope', 'preview'], check=True)

## 预览参考效果
服务仅绑定板端 localhost。在 VS Code 的端口面板转发远程端口 8870，打开该行的本地地址，并追加 `/digimouse/`。使用终端 SSH 转发时连接你已有的主机别名，以实际本地端口为准。重复执行本单元格不会重复启动服务。该步骤不操作 FPGA。

In [ ]:
import functools, http.server, threading
if 'viewer_server' not in globals():
    handler = functools.partial(http.server.SimpleHTTPRequestHandler, directory=str(ROOT / 'preview'))
    viewer_server = http.server.ThreadingHTTPServer(('127.0.0.1', 8870), handler)
    threading.Thread(target=viewer_server.serve_forever, daemon=True).start()
print('参考模型: http://localhost:8870/digimouse/')

## 实板检查和对照
必须在 KV260 的 Kria-PYNQ 内核执行。以下单元格编译 ARM 本机库，加载交付 overlay，并测试解析球与 Digimouse 心脏标注 ROI 的单/双核结果。标注 ROI 是派生连续场，不是原始 CT 灰度。若访问 PL 权限不足，使用正确权限重新启动 Jupyter；不要把密码或凭据写进 Notebook。

In [ ]:
subprocess.run([sys.executable, 'scripts/check_environment.py'], check=True)
subprocess.run(['bash', 'scripts/install.sh'], check=True, env=dict(os.environ, PYTHON=sys.executable))
subprocess.run([sys.executable, 'scripts/board_test.py'], check=True)
import json
board = json.loads((ROOT / 'output/board_report.json').read_text())
assert board['passed'] and board['board_tested']
board

## 重建小鼠全身解剖
Digimouse 配置重建体表、骨骼和 14 个内部图层。默认使用双核、每缓冲 32 MiB 预算。完整网格流式写入存储。程序只在完整网格哈希与参考完全一致后复用显示 LOD。

In [ ]:
profile = 'digimouse'
subprocess.run([sys.executable, 'scripts/reconstruct_scene.py', '--backend', 'pynq',
    '--data', str(ROOT / 'data' / profile), '--output', str(ROOT / 'output' / profile),
    '--buffer-mb', '32', '--engines', '2'], check=True)
actual = json.loads((ROOT / 'output' / profile / 'scene.json').read_text())
assert actual['backend'] == 'pynq' and actual['board_tested']
assert all(x['display_verified_against_full'] for x in actual['layers'])
actual

In [ ]:
# 停止参考服务，切换为本次实际重建结果。
if 'viewer_server' in globals():
    viewer_server.shutdown(); viewer_server.server_close(); del viewer_server
handler = functools.partial(http.server.SimpleHTTPRequestHandler, directory=str(ROOT / 'output'))
viewer_server = http.server.ThreadingHTTPServer(('127.0.0.1', 8870), handler)
threading.Thread(target=viewer_server.serve_forever, daemon=True).start()
print(f'PYNQ 本次结果: http://localhost:8870/{profile}/')